
#### Bronze XML Ingestion script:

In the Medallion architecture, the Bronze layer must house 100% of the raw data. Since large files were split into different formats to optimize driver memory, this script acts as the specific handler for the XML portion.


In [0]:
import os
import json
from pyspark.sql import functions as F

# --- 1. CONFIGURATION & WIDGETS ---
# Define a widget to accept the list of datasets to process via JSON array
dbutils.widgets.text("datasets_json", "[]", "Datasets List (JSON Array)")

try:
    # Retrieve and parse the dataset list from the widget
    raw_input = dbutils.widgets.get("datasets_json")
    dataset_list = json.loads(raw_input)
    
    # Standard pathing for Landing Volumes and Bronze Catalog
    SOURCE_BASE = "/Volumes/data_landing/data_raw"
    DEST_CATALOG = "data_bronze"
    DEST_SCHEMA = "bronze"
    
except Exception as e:
    print(f"Setup Error: {str(e)}")
    raise

# --- 2. MODULAR XML INGESTION COMPONENT ---

def ingest_xml_to_bronze(dataset_name):
    """
    Locates and ingests 'chunk4.xml' from the landing volume and 
    appends the records to the existing Bronze Delta table for the dataset.
    """
    try:
        # Define paths for the source XML and the target Bronze Delta table
        source_file = f"{SOURCE_BASE}/{dataset_name}/chunks/chunk4.xml"
        target_table = f"{DEST_CATALOG}.{DEST_SCHEMA}.{dataset_name.lower()}"
        
        # Internal variable for Spark XML loading
        XML_PATH = source_file
        
        # Check if the XML chunk exists before attempting to start a Spark job
        if not os.path.exists(source_file):
            print(f"Skipping: {dataset_name} (No XML chunk found at {source_file})")
            return

        print(f"Ingesting XML: {dataset_name} -> {target_table}")

        # 2. READ XML WITH EXPLICIT NESTING
        # Uses the 'spark-xml' connector logic to parse tags into columns
        df_xml = (spark.read
                  .format("xml")
                  .option("rowTag", "item") # Primary target tag for records
                  .load(XML_PATH))

        # 3. ROBUST TAG INFERENCE
        # Validates if the 'item' tag yielded data; if not, falls back to 'Record'
        if len(df_xml.columns) == 0 or df_xml.count() == 0:
            print(f"!!! WARNING: No records found in {source_file} using 'item' tag.")
            print("Retrying with broader inference using 'Record' tag...")
            df_xml = spark.read.format("xml").option("rowTag", "Record").load(XML_PATH)

        # 4. DATA STANDARDIZATION & AUDITING
        # Bronze layer requirement: Cast all values to string to prevent schema mismatch
        # withColumn adds audit metadata for Data Quality monitoring
        final_df = (df_xml.select([F.col(c).cast("string") for c in df_xml.columns])
                          .withColumn("load_dt", F.current_timestamp())
                          .withColumn("source", F.lit("chunk4.xml")))

        row_count = final_df.count()
        print(f"Verified rows for {dataset_name}: {row_count}")

        # 5. APPEND TO DELTA TABLE
        # mergeSchema is enabled to handle any unique columns present in the XML
        (final_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(target_table))
        
        print(f"SUCCESS: Appended XML data to {target_table}")

    except Exception as e:
        print(f"ERROR: Failed processing {dataset_name}: {str(e)}")

# --- 3. EXECUTION ORCHESTRATOR ---
if __name__ == "__main__":
    if not dataset_list:
        print("No datasets provided for XML ingestion.")
    else:
        # Iteratively process each dataset provided in the JSON list
        for ds in dataset_list:
            ingest_xml_to_bronze(ds)

### Unit testing:

Consistent with the Data Quality arrow in your architecture, this script acts as a gatekeeper before data moves toward the Silver tier. It ensures that all data in the Bronze layer remains in its rawest form—represented as strings—to prevent ingestion failures during the initial "Raw data ingestion" phase.|

In [0]:
def validate_bronze_layer(target_datasets):
    """
    Standardizes validation across all ingested Bronze tables.
    Validates record counts, string-only typing, and audit columns.
    """
    print(f"Validation started for {len(target_datasets)} datasets.")
    print("-" * 60)
    
    for ds in target_datasets:
        # Construct the fully qualified table name for the Bronze Delta table
        table_full_name = f"{DEST_CATALOG}.{DEST_SCHEMA}.{ds.lower()}"
        
        try:
            # 1. Verify Table Existence: Ensure the Delta table was successfully created
            assert spark.catalog.tableExists(table_full_name), f"Table {table_full_name} not found."
            
            # 2. Scope Validation: Filter specifically for records originating from the XML ingestion
            # This allows us to verify the success of the most recent 'chunk4.xml' load
            test_df = spark.table(table_full_name).filter(F.col("source") == "chunk4.xml")
            row_count = test_df.count()
            
            # 3. Validation: Data Presence
            # Ensures that the XML ingestion phase actually produced data in the table
            assert row_count > 0, f"Zero XML records found in {table_full_name}."
            
            # 4. Validation: Column Data Types (Bronze Layer Requirement)
            # The Bronze tier must store raw data as strings to prevent schema drift issues
            # Ensure every column is String except the system-generated load_dt timestamp
            for field in test_df.schema:
                if field.name != "load_dt":
                    assert str(field.dataType) == "StringType()", \
                        f"Type mismatch in {ds}: {field.name} is {field.dataType}, expected StringType."
            
            # 5. Validation: Audit Columns
            # Confirms existence of metadata required for lineage and Data Quality tracking
            current_cols = test_df.columns
            assert "load_dt" in current_cols and "source" in current_cols, \
                f"Missing audit columns in {ds}."
            
            print(f"PASSED: {ds.ljust(25)} | Rows: {row_count}")
            
        except AssertionError as ae:
            # Captured assertion failures indicate a logic or data issue during ingestion
            print(f"FAILED: {ds.ljust(25)} | Reason: {str(ae)}")
            # Note: Non-chunked datasets will fail here because 'chunk4.xml' won't exist in their source column
        except Exception as e:
            # Captures technical Spark or environment errors
            print(f"ERROR:  {ds.ljust(25)} | Technical Error: {str(e)[:50]}")

    print("-" * 60)
    print("Bronze validation process completed.")

# --- EXECUTION ---
if __name__ == "__main__":
    if dataset_list:
        validate_bronze_layer(dataset_list)
    else:
        print("No datasets provided for validation.")

# Validation failed for datasets which were not chunked and 2 are chunked so they pass the condition